This script only runs from within Meteowiss, because of the data store. It compiles raw data files stored on MeteoSwiss disk into parquet files. 
Incoming raw data files are first organized into folders, and a stastic is computed and displayed showing the number of recently incoming files.
Then, yearly files are generated for Meteo bulletins, Thermo zip files, NOAA CPD2 tarballs. Monthly files are generated for AE33 zip files, G2401 tarballs.

joerg.klausen@meteoswiss.ch

[TODO] Improve handling of erroneous files from g2401.compile_g2401_to_parquet

In [ ]:
import os
import yaml
import housekeeping.organize_files as hk
import monitoring.file_coverage as fc
from processing.meteo import Meteo
from processing.thermo import Thermo
from processing.cpd2 import CPD2
from processing.ae33 import AE33
from processing.g2401 import G2401

# read configuration
with open("mch-config.yml", "r") as fh:
    cfg = yaml.safe_load(fh)
    fh.close()

mkn = cfg["mkn"]
# nrb = cfg['nrb']

In [ ]:
# organize MKN files on MeteoSwiss fileshare
n = hk.organize_files(mkn, branch="incoming", verbosity=1)

# show and plot file statistics
days = 7
stats = fc.get_file_coverage(cfg=mkn, days=days)
display(df:=fc.print_coverage(stats=stats, days=days))
fc.plot_coverage(stats=stats, days=days)

In [ ]:
# relative path of level1 data .parquet files on repo
level1 = os.path.join("data", "level1")
os.makedirs(level1, exist_ok=True)

for k, v in mkn['branches'].items():
    if not os.path.exists(os.path.join(mkn['root'], v)):
        raise ValueError(f"path '{v}' not found.")

# path of source files
incoming = os.path.join(mkn['root'], mkn['branches']['incoming'])

# path of archive for successfully processed raw data files.
archive = os.path.join(mkn['root'], mkn['branches']['archive'])

# path for raw data files with issues
issues = os.path.join(mkn['root'], mkn['branches']['issues'])

# path for log files
logs = os.path.join(mkn['root'], mkn['branches']['logs'])

# instantiate instruments (rather, data types)
met = Meteo(log=os.path.join(logs, "meteo.log"))
thermo = Thermo(log=os.path.join(logs, "thermo.log"))
cpd2 = CPD2(log=os.path.join(logs, "cpd2.log"))
ae33 = AE33(log=os.path.join(logs, "ae33.log"))
g2401 = G2401(log=os.path.join(logs, "g2401.log"))

In [ ]:
# process all raw data types
years = [f"{year}" for year in range(2021, 2025)]
months = ["{:02d}".format(mm) for mm in range(1, 13, 1)]

for year in years:
    met.compile_vrxa00_to_parquet(
        source=os.path.join(incoming, "meteo", year), 
        target=os.path.join(level1, year), 
        archive=os.path.join(archive, "meteo", year), 
        issues=os.path.join(issues, "meteo"),
        )
    thermo.compile_thermo_to_parquet(
        source=os.path.join(incoming, "tei49c", year),
        target=os.path.join(level1, year),
        archive=os.path.join(archive, "tei49c", year),
        issues=os.path.join(issues, "tei49c"),
        )        
    thermo.compile_thermo_to_parquet(
        source=os.path.join(incoming, "tei49i", year),
        target=os.path.join(level1, year),
        archive=os.path.join(archive, "tei49i", year),
        issues=os.path.join(issues, "tei49i"),
        )
    cpd2.tarballs_to_parquet(
        source=os.path.join(incoming, "aerosol", year), 
        target=os.path.join(level1, year), 
        archive=os.path.join(archive, "aerosol", year), 
        issues=os.path.join(issues, "aerosol"),
        )
    for month in months:
        if os.path.exists(incoming):
            ae33.zipfiles_to_parquet(
                source=os.path.join(incoming, "ae33", "data", year, month), 
                target=os.path.join(level1, year, month), 
                archive=os.path.join(archive, "ae33", "data", year, month), 
                issues=os.path.join(issues, "ae33"), 
                plot=True,
                )
            g2401.compile_g2401_to_parquet(
                source=os.path.join(incoming, "g2401", year, month), 
                target=os.path.join(level1, year, month), 
                archive=os.path.join(archive, "g2401", year, month), 
                issues=os.path.join(issues, "g2401", year),
                )

In [ ]:
# display empty directories under source
# os.system(f"find {source} -empty -type d)

# display empty directories under source (NB: no questions asked!)
# os.system(f"find {source} -empty -type d -delete")

In [ ]:
nrb = cfg["nrb-brewer"]
# n = hk.organize_files(nrb)
# print(f"Finished organizing files under '{nrb['root']}'. {n} files moved.")

days = 7
stats = fc.get_file_coverage(cfg=nrb, days=days)
fc.plot_coverage(stats=stats, days=days)
fc.print_coverage(stats=stats, days=days)

In [ ]:
nrb = cfg["nrb-dobson"]
n = hk.organize_files(nrb, branch="uploads")
print(f"Finished organizing files under '{nrb['root']}'. {n} files moved.")

days = 60
stats = fc.get_file_coverage(cfg=nrb, days=days)
fc.plot_coverage(stats=stats, days=days)
fc.print_coverage(stats=stats, days=days)
